# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [34]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
#!pip install groq -q #Intalamos la librería de groq

import os
from groq import Groq #Mandar a Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('API_GROQ'))
print ("Cliente de Qroq inicializado cporrectamente.")

Cliente de Qroq inicializado cporrectamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [35]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿CUANTAS PERSONAS VIVEN EN MEXICO" #"¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

print(prompt)


¿CUANTAS PERSONAS VIVEN EN MEXICO


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [36]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt}]
    )
print(response.choices[0].message.content)


La población de México se sitúa alrededor de **126 millones de personas**.  

- **Censo 2020 (INEGI):** 126 014 024 habitantes.  
- **Estimación 2023 (World Bank/INEGI):** ≈ 126,5 millones.

El país sigue creciendo a una tasa de aproximadamente 1 % anual, por lo que la cifra actual es un poco mayor que la del censo.


In [37]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-498424d7-a3ea-4306-bcf8-5cc0aaf6ad85",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La población de México se sitúa alrededor de **126 millones de personas**.  \n\n- **Censo 2020 (INEGI):** 126 014 024 habitantes.  \n- **Estimación 2023 (World Bank/INEGI):** ≈ 126,5 millones.\n\nEl país sigue creciendo a una tasa de aproximadamente 1 % anual, por lo que la cifra actual es un poco mayor que la del censo.",
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": "The user: \"¿CUANTAS PERSONAS VIVEN EN MEXICO\" Spanish. They ask: \"How many people live in Mexico?\" They want a population figure. I should provide current estimate. The latest census data: 2020 census population: about 126,014,024. But the user may want a current estimate. According to World Bank, 2023 estimate about 126.7 million. Th

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [38]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print(f"Tokens del prompt: {response.usage.prompt_tokens}")
print(f"Tokens de la respuesta: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")



Tokens del prompt: 83
Tokens de la respuesta: 236
Total tokens: 319


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [39]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time
inicio = time.time()
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt}]
    )
duracion =time.time() - inicio
#duracion_ms = duracion*1000 #para ms

print(f"Tiempo de respuesta: {duracion:.2f} segundos")
#print(f"Tiempo de respuesta: {duracion_ms:.2f} milisegundos





Tiempo de respuesta: 0.59 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [40]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role":"user", "content":prompt}]
    )
duracion_grande =time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s - {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s - {response_grande.usage.total_tokens} tokens")
print("\nREspuesta del modelo grande:",response_grande.choices[0].message.content)



Modelo ligero: 0.59 s - 543 tokens
Modelo grande: 0.83 s - 400 tokens

REspuesta del modelo grande: Según los datos más recientes de instituciones como el Instituto Nacional de Estadística y Geografía (INEGI) y el Banco Mundial, la población de México a mediados de 2026 se estima en **aproximadamente 130 millones de habitantes**.  

- **2023:** ~126 millones (censo y proyecciones oficiales).  
- **2024‑2025:** crecimiento anual de alrededor 0,9 % (≈1,1 millones de personas por año).  
- **2026 (estimación):** 126 M + 2 años × 1,1 M ≈ 128,2 M; redondeado a 130 M para reflejar la mayor precisión de fuentes internacionales que ya incluyen migración y nacimientos recientes.

Ten en cuenta que estas cifras son estimaciones; el próximo censo oficial (programado para 2027) proporcionará el número exacto. Si necesitas datos más detallados (por estado, edad, etc.) házmelo saber.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [41]:
# Leer API key desde Colab Secrets

client = Groq(api_key=userdata.get('API_GROQ'))
print ("Cliente de Qroq inicializado cporrectamente.")

Cliente de Qroq inicializado cporrectamente.


In [42]:
# Definir la lista de preguntas
prompt1 = "Diferencia entre alzheimer y demencia"
prompt2 = "¿Cuántas universidades hay en México?"
prompt3 = "¿Quién es el actual presidente de Panamá"

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [43]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
#Tiempo de respuesta
inicio = time.time()
response1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt1}]
    )
respuesta1 = response1.choices[0].message.content

duracion1 =time.time() - inicio

Tokens_prompt1 = response1.usage.prompt_tokens
Tokens_respuesta1 = response1.usage.completion_tokens
Total_tokens1 = response1.usage.total_tokens

# Guardar todo en un diccionario
resultado_1 = {
    "Respuesta": respuesta1,
    "\nTiempo_respuesta_segundos": duracion1,
    "\nTokens_prompt": Tokens_prompt1,
    "\nTokens_respuesta": Tokens_respuesta1,
    "\nTokens_totales": Total_tokens1
}

# Mostrar el resultado
print("Resultado 1:")
for clave, valor in resultado_1.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 1:
Respuesta: ## Demencia vs. Alzheimer

| **Característica** | **Demencia (síndrome)** | **Alzheimer (enfermedad específica)** |
|--------------------|------------------------|--------------------------------------|
| **Definición** | Grupo de síntomas que indican deterioro cognitivo suficientemente severo como para interferir con la vida diaria. | Enfermedad neurodegenerativa progresiva, la causa más frecuente de demencia. |
| **Causa** | Puede ser de origen múltiple: Alzheimer, demencia vascular, demencia con cuerpos de Lewy, demencia frontotemporal, demencia por infección, por deficiencia de vitaminas, por trastornos endocrinos, etc. | Acumulación anormal de **placas de β‑amilo** y **ovillos neurofibrilares** (tangentes tau) en el cerebro. |
| **Edad de aparición típica** | Varía según la causa: Alzheimer suele aparecer después de los 65 años; demencia vascular puede surgir en 60‑70; demencia frontotemporal a veces a los 45‑65. | A partir de los 65 años (pauta clásica), p

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [44]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
#Tiempo de respuesta
inicio = time.time()
response2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt2}]
    )
respuesta2 = response2.choices[0].message.content

duracion2 = time.time() - inicio

Tokens_prompt2 = response2.usage.prompt_tokens
Tokens_respuesta2 = response2.usage.completion_tokens
Total_tokens2 = response2.usage.total_tokens

# Guardar todo en un diccionario
resultado_2 = {
    "Respuesta": respuesta2,
    "\nTiempo_respuesta_segundos": duracion2,
    "\nTokens_prompt": Tokens_prompt2,
    "\nTokens_respuesta": Tokens_respuesta2,
    "\nTokens_totales": Total_tokens2
}

# Mostrar el resultado
print("Resultado 2:")
for clave, valor in resultado_2.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 2:
Respuesta: En México existen **aproximadamente 1 170 instituciones de educación superior** (IES).  
Si hablamos específicamente de *universidades*—instituciones que ofrecen estudios de licenciatura, posgrado y doctorado—el número es mucho menor.

Según el **Instituto Nacional de Evaluación de la Educación Superior (INEES) y la Secretaría de Educación Pública (SEP)**, en el ejercicio 2023 las instituciones que cumplen con la definición de universidad son:

| Tipo de universidad | Número |
|----------------------|--------|
| **Federales** (universidades bajo administración de la Federación) | 73 |
| **Estatales** (universidades bajo administración de los gobiernos estatales) | 38 |
| **Privadas** (universidades privadas con reconocimiento oficial) | 65 |
| **Total** | **176** |

> **Resumen**  
> • **Total de IES en México (2023):** 1 170  
> • **Universidades (federales + estatales + privadas):** 176  

Esta cifra incluye a la Universidad Nacional Autónoma de México (UNAM),

In [45]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
#Tiempo de respuesta
inicio = time.time()
response3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt3}]
    )
respuesta3 = response3.choices[0].message.content

duracion3 = time.time() - inicio

Tokens_prompt3 = response3.usage.prompt_tokens
Tokens_respuesta3 = response3.usage.completion_tokens
Total_tokens3 = response3.usage.total_tokens

# Guardar todo en un diccionario
resultado_3 = {
    "Respuesta": respuesta3,
    "\nTiempo_respuesta_segundos": duracion3,
    "\nTokens_prompt": Tokens_prompt3,
    "\nTokens_respuesta": Tokens_respuesta3,
    "\nTokens_totales": Total_tokens3
}

# Mostrar el resultado
print("Resultado 3:")
for clave, valor in resultado_3.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 3:
Respuesta: El presidente actual de Panamá es **José Luis Gutiérrez**.  

- **Partido político**: Partido Revolucionario Democrático (PRD).  
- **Fecha de toma de posesión**: 1 de octubre de 2024.  

Gutiérrez ganó las elecciones presidenciales de 2024 y ha estado ejerciendo el cargo desde entonces. Si necesitas más información sobre su administración o alguna otra consulta, avísame.

Tiempo_respuesta_segundos: 0.8533737659454346

Tokens_prompt: 79

Tokens_respuesta: 677

Tokens_totales: 756


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [46]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)
print(resultados)

[{'Respuesta': '## Demencia vs. Alzheimer\n\n| **Característica** | **Demencia (síndrome)** | **Alzheimer (enfermedad específica)** |\n|--------------------|------------------------|--------------------------------------|\n| **Definición** | Grupo de síntomas que indican deterioro cognitivo suficientemente severo como para interferir con la vida diaria. | Enfermedad neurodegenerativa progresiva, la causa más frecuente de demencia. |\n| **Causa** | Puede ser de origen múltiple: Alzheimer, demencia vascular, demencia con cuerpos de Lewy, demencia frontotemporal, demencia por infección, por deficiencia de vitaminas, por trastornos endocrinos, etc. | Acumulación anormal de **placas de β‑amilo** y **ovillos neurofibrilares** (tangentes tau) en el cerebro. |\n| **Edad de aparición típica** | Varía según la causa: Alzheimer suele aparecer después de los 65 años; demencia vascular puede surgir en 60‑70; demencia frontotemporal a veces a los 45‑65. | A partir de los 65 años (pauta clásica), per

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [47]:
#Mostrar resultados
for resultado in resultados:
    print(resultado)

#Comprobar si resolvió correctamente las 3 preguntas
if all(resultados):
    print("El modelo ligero resolvió las 3 preguntas satisfactoriamente.")
else:
    print("El modelo ligero no resolvió las 3 preguntas satisfactoriamente.")

{'Respuesta': '## Demencia vs. Alzheimer\n\n| **Característica** | **Demencia (síndrome)** | **Alzheimer (enfermedad específica)** |\n|--------------------|------------------------|--------------------------------------|\n| **Definición** | Grupo de síntomas que indican deterioro cognitivo suficientemente severo como para interferir con la vida diaria. | Enfermedad neurodegenerativa progresiva, la causa más frecuente de demencia. |\n| **Causa** | Puede ser de origen múltiple: Alzheimer, demencia vascular, demencia con cuerpos de Lewy, demencia frontotemporal, demencia por infección, por deficiencia de vitaminas, por trastornos endocrinos, etc. | Acumulación anormal de **placas de β‑amilo** y **ovillos neurofibrilares** (tangentes tau) en el cerebro. |\n| **Edad de aparición típica** | Varía según la causa: Alzheimer suele aparecer después de los 65 años; demencia vascular puede surgir en 60‑70; demencia frontotemporal a veces a los 45‑65. | A partir de los 65 años (pauta clásica), pero